# Distil the teacher into the 0.8B: SFT on an A100 or an L4

One notebook for each round of distillation, top to bottom on one GPU. Round 4 triples the teacher data: round 3's recipe, on the teacher's sessions over two train suites instead of one. The teacher half runs once per suite; a later round finds the sessions and pairs on Drive and goes straight to training:

1. **The teacher on the train suites.** `Qwen/Qwen3.8-27B-FP8` on vLLM, with the settings of its `dev` baseline (`qwen3.8-27b-fp8-v5`), over generated cases whose users have half their merchants and item names invented for them, so the student learns to read a receipt rather than recognise a shop. Both suites are git-ignored, regenerated here and checked against their fingerprints:
   - `train`: 2,100 cases for 210 users (`make_suite.py --name train --per-locale 300 --users-per-locale 30 --seed 1 --novel-share 0.5`), the teacher run rounds 1–3 trained on (`teacher-train`, already on Drive);
   - `train-2`: 4,200 new cases for 420 new users (`--name train-2 --per-locale 600 --users-per-locale 60 --seed 4 --novel-share 0.5`), new this round (`teacher-train-2`).

   Never `dev`, `dev-novel` or `holdout`: `sft_data.py` refuses them. Then the teacher on `dev-novel` (280 cases, every merchant and item invented, from the half of the invented names no train suite can draw), if its row isn't on Drive yet.
2. **The training pairs.** `sft_data.py` takes both runs into one file. It keeps the sessions whose case graded `record_all`, rebuilds every request the agent sent in them, and renders each as a (prompt, completion) pair with Qwen3.5-0.8B's chat template, thinking off: tool calls, the end of the tool phase (an empty reply) and the answer JSON. The sessions that drafted a wrong record add their answer with its confidence set to `DOUBT_CONFIDENCE`, only that number trained on, as in round 3. New this round: a train case whose content a frozen suite repeats word for word is left out (`frozen_content` in the stats: about 170 of `train`, 340 of `train-2`, all non-spending messages from the generators' small pool), as `train_grpo.py` already does. Rounds 1–3 trained on them, so their `non_spending` scores may be partly memory, and round 4's may be a little lower for that reason.
3. **SFT.** vLLM is stopped and `train_sft.py` trains a fresh LoRA from the base model on the same GPU: every linear layer including the Gated DeltaNet projections, loss on the completion only, averaged per token.
4. **Merge.** `merge_lora.py` adds the adapter into a copy of the original checkpoint, so the result has the base model's layout (`Qwen3_5ForConditionalGeneration`) that llama.cpp's converter reads.
5. **Score it unquantized.** The merged model at bf16 on vLLM over `dev` and `dev-novel`, against SFT round 3, GRPO round 2 and the teacher.

| | teacher pass | SFT |
|---|---|---|
| model | `Qwen/Qwen3.8-27B-FP8` | `Qwen/Qwen3.5-0.8B` + a fresh LoRA |
| server / trainer | vLLM, 16 cases in flight (`--max-num-seqs 32`), thinking off, 10 tool rounds / 800 / 1,500 tokens | `train_sft.py`, bf16, rank 16, 1 epoch, lr 2e-4, 8 sequences a step, `--max-len 8192`, loss per token |
| time | `train`: done (77 min on an A100 80GB); `train-2`: about 2.6 hours; `dev-novel`: about 12 min | about 6,400 pairs took 75 min in round 3; round 4 has about three times as many, so about 4 hours on an A100 |

The teacher needs **Runtime → Change runtime type → A100 GPU**: the 27B teacher doesn't fit in less than 40 GB. Once the teacher's sessions and pairs are on Drive, an **L4** runs the training: the notebook sees the teacher's work is done, never starts the teacher, and only trains and scores the student. It stops early if a teacher run is still needed and the GPU is too small. A T4 doesn't work for SFT (no bf16, and the DeltaNet kernels are slow there). The notebook clones `REPO_URL` at `BRANCH`, [kurexi/cost](https://github.com/kurexi/cost) by default; point them at a fork to run your own changes (for a private one, add a Colab secret named `GITHUB_TOKEN` with read access to it). `AGENT` is the agent's folder inside the clone: empty for kurexi/cost, where the agent is the whole repository. `RECORDS` is the folder, inside the clone, that holds `runs/` and `RESULTS.md`; `eval-records` by default, the repository's own runs; leave it empty to keep them in the agent's `eval/` folder. The eval scripts find it through `SPENDING_EVAL_RECORDS`.

**Everything that takes long is kept on Google Drive** (`MyDrive/spending-agent/`): the teacher runs, the pairs, the adapter with its checkpoints and the merged model. If Colab disconnects, run the cells again from the top: a teacher run resumes from its last finished case, training from its last checkpoint, and finished steps are skipped. The merged model stays on Drive for the GRPO notebook (`grpo.ipynb`, `SFT_ROUND`) and the T4 notebook (`baseline_gguf_t4.ipynb`, `SOURCE`).

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
from google.colab import userdata

def sh(args, cwd=None):
    """Run a command with its output in the cell: a process started by
    subprocess.run prints to the runtime's log, which Colab doesn't show."""
    process = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)


REPO_URL = "https://github.com/kurexi/cost"
BRANCH = "main"
RECORDS = "eval-records"
AGENT = ""
CLONE = "/content/repo"
EVAL = os.path.join(CLONE, AGENT, "eval")
RECORDS_DIR = f"{CLONE}/{RECORDS}" if RECORDS else EVAL
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = ""
url = REPO_URL.replace("https://", f"https://{token}@", 1) if token else REPO_URL
os.chdir("/content")
sh(["rm", "-rf", CLONE])
sh(["git", "clone", "--depth", "1", "--branch", BRANCH, url, CLONE])
os.environ["SPENDING_EVAL_RECORDS"] = RECORDS_DIR
os.makedirs(f"{RECORDS_DIR}/runs", exist_ok=True)

## Settings

`ROUND` names everything this notebook makes: the adapter, the merged model (`qwen3.5-0.8b-sft-r4`) and its runs. Change it for a new round, so a round never overwrites another on Drive. `PREVIOUS` is the round its scores are compared with, and `GRPO_REFERENCE` the best GRPO model so far, shown beside them.

`TEACHER_RUNS` lists the teacher's runs the pairs are built from, each with its train suite, the arguments that generate it, its fingerprint and its size. Adding a suite is adding an entry: a run not finished on Drive is run, the others are reused.

`LOSS` is how the completion loss is averaged: per token since round 2, so a one-token stop and a 25-token tool call weigh the same per token. `DOUBT_CONFIDENCE` is round 3's change, kept: the teacher's wrong drafts teach the student this confidence, below the 0.8 auto-approval bar, and nothing else.

In [ ]:
TEACHER = "Qwen/Qwen3.8-27B-FP8"
TEACHER_RUNS = {
    "teacher-train": {"suite": "train", "fingerprint": "a5437ef135b8f48f", "cases": 2100,
                      "args": ["--per-locale", "300", "--users-per-locale", "30", "--seed", "1", "--novel-share", "0.5"]},
    "teacher-train-2": {"suite": "train-2", "fingerprint": "9c208ad58ed4d37a", "cases": 4200,
                        "args": ["--per-locale", "600", "--users-per-locale", "60", "--seed", "4", "--novel-share", "0.5"]},
}
STUDENT = "Qwen/Qwen3.5-0.8B"
ROUND = "r4"
PREVIOUS = "qwen3.5-0.8b-sft-r3"
GRPO_REFERENCE = "qwen3.5-0.8b-grpo-r2"
LOSS = "token"
DOUBT_CONFIDENCE = 0.5
NAME = f"qwen3.5-0.8b-sft-{ROUND}"
TEACHER_NAME = "qwen3.8-27b-fp8-v5"
EPOCHS = 1
RANK = 16
LR = 2e-4
ACCUM = 8
MAX_LEN = 8192
print(NAME)

## Drive

The teacher's runs under `runs/` are links into Drive, so `run.py` appends each finished case there and a rerun resumes from it. The cell then says which teacher runs are still to do: none, once the pairs are on Drive and the teacher has finished `dev-novel`.

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/spending-agent"
TEACHER_DIRS = {run: f"{DRIVE}/runs/{run}" for run in TEACHER_RUNS}
TEACHER_NOVEL_RUN = f"{TEACHER_NAME}-dev-novel"
TEACHER_NOVEL_DIR = f"{DRIVE}/runs/{TEACHER_NOVEL_RUN}"
PAIRS = f"{DRIVE}/sft/{'+'.join(TEACHER_RUNS)}" + (f"-doubt{DOUBT_CONFIDENCE:g}" if DOUBT_CONFIDENCE is not None else "") + ".jsonl"
ADAPTER = f"{DRIVE}/sft/{NAME}"
MERGED = f"{DRIVE}/models/{NAME}"
for folder in (*TEACHER_DIRS.values(), TEACHER_NOVEL_DIR, os.path.dirname(PAIRS), ADAPTER, os.path.dirname(MERGED)):
    os.makedirs(folder, exist_ok=True)


def keep_on_drive(run, kept):
    local = f"{RECORDS_DIR}/runs/{run}"
    if os.path.islink(local):
        return
    if os.path.isdir(local) and not os.listdir(kept):
        shutil.copytree(local, kept, dirs_exist_ok=True)
    sh(["rm", "-rf", local])
    os.symlink(kept, local)


def finished(folder, cases):
    results = f"{folder}/results.jsonl"
    return os.path.exists(results) and sum(1 for _ in open(results)) >= cases


for run, kept in TEACHER_DIRS.items():
    keep_on_drive(run, kept)
keep_on_drive(TEACHER_NOVEL_RUN, TEACHER_NOVEL_DIR)
gpu_mib = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.split()[0])
TEACHER_FITS = gpu_mib >= 39000
NEEDS_TRAIN_RUNS = [] if os.path.exists(PAIRS) else [
    run for run, spec in TEACHER_RUNS.items() if not finished(TEACHER_DIRS[run], spec["cases"])]
NEEDS_NOVEL_RUN = not finished(TEACHER_NOVEL_DIR, 280)
if NEEDS_TRAIN_RUNS and not TEACHER_FITS:
    raise SystemExit(f"the teacher runs {NEEDS_TRAIN_RUNS} aren't finished and this GPU has {gpu_mib} MiB: use an A100")
SERVE_TEACHER = TEACHER_FITS and (bool(NEEDS_TRAIN_RUNS) or NEEDS_NOVEL_RUN)
for run in TEACHER_RUNS:
    print(f"teacher run {run}:", "to run" if run in NEEDS_TRAIN_RUNS else "done")
print("teacher dev-novel run:", "done" if not NEEDS_NOVEL_RUN else "to run" if TEACHER_FITS
      else "missing, and this GPU can't host the teacher: the dev-novel comparison will lack its row")

## The train suites

Each generated in a second or two from its seed; the fingerprints show they are the same suites every time. `sft_data.py` reads them too, to leave out the cases whose content a frozen suite repeats. `dev-novel` is committed and frozen like `dev`.

In [ ]:
%cd {EVAL}
import json
for run, spec in TEACHER_RUNS.items():
    subprocess.run(["python3", "make_suite.py", "--name", spec["suite"], *spec["args"]], check=True, capture_output=True)
    with open(f"suites/{spec['suite']}/meta.json") as f:
        suite = json.load(f)
    print(spec["suite"], suite["cases"], "cases, fingerprint", suite["fingerprint"])
    if suite["fingerprint"] != spec["fingerprint"] or suite["cases"] != spec["cases"]:
        raise SystemExit(f"{spec['suite']} doesn't regenerate as it did: the generator changed, so this is a new suite")

## The teacher on vLLM

vLLM goes in its own virtual environment, so its pinned `torch` and `transformers` can't clash with the training half. Same flags as the teacher's `dev` run: the checkpoint is FP8 already, 16,384 tokens per sequence, `--max-num-seqs 32`, the `qwen3_coder` tool parser and the `qwen3` reasoning parser, served as `qwen`. The weights are about 30 GB, so the first start downloads for several minutes. Logs go to `/content/vllm.log`.

vLLM is installed either way: the student is scored on it too. The teacher is only started when the *Drive* cell said a teacher run is still to do.

In [ ]:
VLLM_VENV = "/content/vllm-venv"
if not os.path.exists(f"{VLLM_VENV}/bin/vllm"):
    sh(["pip", "install", "-q", "uv"])
    sh(["uv", "venv", "-q", "--python", "3.12", VLLM_VENV])
    sh(["uv", "pip", "install", "-q", "--python", f"{VLLM_VENV}/bin/python", "vllm"])
print(subprocess.run([f"{VLLM_VENV}/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                     capture_output=True, text=True).stdout)

In [ ]:
import time
import urllib.request


def stop_servers():
    subprocess.run("pkill -f '[v]llm serve'; pkill -f '[l]lama-server'", shell=True)
    for _ in range(60):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.strip()
        if int(used.split()[0]) < 1000:
            return
        time.sleep(5)
    raise SystemExit(f"the GPU still holds {used} MiB")


def log_tail(path, characters=4000):
    with open(path, errors="replace") as f:
        return f.read()[-characters:]


def serve(model, extra=""):
    """Start vLLM on `model` and wait for it, printing its latest log line
    every 30 seconds; stop at once, with the end of the log, if it exits."""
    stop_servers()
    log_path = "/content/vllm.log"
    process = subprocess.Popen([f"{VLLM_VENV}/bin/vllm", "serve", model, "--served-model-name", "qwen",
                                "--max-model-len", "16384", "--max-num-seqs", "32", "--gpu-memory-utilization", "0.90",
                                "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
                                "--reasoning-parser", "qwen3", "--port", "8090", *extra.split()],
                               stdout=open(log_path, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    started = reported = time.time()
    while True:
        if process.poll() is not None:
            raise SystemExit(f"vLLM exited with code {process.returncode}:\n{log_tail(log_path)}")
        try:
            urllib.request.urlopen("http://127.0.0.1:8090/v1/models", timeout=5)
            print(f"server is up after {time.time() - started:.0f} s")
            return
        except Exception:
            pass
        if time.time() - started > 2400:
            raise SystemExit(f"vLLM isn't up after 40 minutes:\n{log_tail(log_path)}")
        if time.time() - reported >= 30:
            lines = log_tail(log_path, 1000).strip().splitlines()
            print(f"{time.time() - started:.0f} s, still starting: {lines[-1] if lines else '(no log yet)'}", flush=True)
            reported = time.time()
        time.sleep(2)

In [ ]:
if SERVE_TEACHER:
    serve(TEACHER)
else:
    print("the teacher isn't needed")

## Check the harness and the answer format

`oracle` must grade 1.000 and `silent` 0 (on a 70-case prefix of `train`). The probe must accept the tool round and the full bounded schema, as it did for the teacher's `dev` run; the cell stops otherwise.

In [ ]:
os.chdir(EVAL)
if SERVE_TEACHER:
    for name, backend in (("check-oracle", "oracle"), ("check-silent", "silent")):
        graded = subprocess.run(["python3", "run.py", "--suite", "train", "--name", name, "--backend", backend,
                                 "--limit", "70", "--restart"], capture_output=True, text=True).stdout
        print(name, [line for line in graded.splitlines() if line.startswith("record_core")])
    probe = subprocess.run(["python3", "probe_formats.py", "--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen"],
                           capture_output=True, text=True)
    print(probe.stdout, probe.stderr)
    verdicts = dict((line[:32].strip(), line[32:].strip()) for line in probe.stdout.splitlines() if line.strip())
    for required in ("tool round (tool_choice auto)", "full bounded schema"):
        if not verdicts.get(required, "").startswith("accepted"):
            raise SystemExit(f"the server refused the {required}: stop here")

## The teacher runs

Each train suite still to do (`train-2`: 4,200 cases, about 2.6 hours on an A100), 16 in flight, thinking off, the default limits: the teacher's `dev` settings. Resumable: if it stops, rerun the *Drive*, *vLLM* and this cell. The grade at the end of each says how many sessions are right enough to train on (`record_all`).

In [ ]:
os.chdir(EVAL)
for run in NEEDS_TRAIN_RUNS:
    sh(["python3", "run.py", "--suite", TEACHER_RUNS[run]["suite"], "--name", run, "--base-url", "http://127.0.0.1:8090/v1",
        "--model", "qwen", "--reasoning-effort", "none", "--workers", "16"])
if not NEEDS_TRAIN_RUNS:
    print("done already")

## The teacher on `dev-novel`

The same teacher and settings on the frozen suite of invented merchants and items, kept on Drive like the `train` run (it is evaluation only, and not in git until it has been committed). Its row on the board is what the student's `dev-novel` score is measured against.

In [ ]:
os.chdir(EVAL)
if SERVE_TEACHER and NEEDS_NOVEL_RUN:
    sh(["python3", "run.py", "--suite", "dev-novel", "--name", TEACHER_NOVEL_RUN, "--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen", "--reasoning-effort", "none", "--workers", "16"])
else:
    print("done already" if not NEEDS_NOVEL_RUN else "skipped: the teacher doesn't fit on this GPU")

## Stop vLLM

Frees the GPU for training, and waits until it is free.

In [ ]:
stop_servers()
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## The training pairs

Training runs in Colab's own Python: `transformers` new enough for Qwen3.5, `peft`, and the `flash-linear-attention` kernels for the DeltaNet layers (without them the delta rule runs on a slow reference implementation). Colab's own `torchao` is removed: training doesn't use it, and `peft` refuses to attach a LoRA while an older version than it supports is installed. The student's checkpoint is downloaded once to local disk.

`sft_data.py` checks its template renderer against `transformers`' `apply_chat_template` on every request of 20 sessions before it writes anything, counts tokens, and drops any pair over `MAX_LEN` instead of letting the trainer cut the instructions off its start. Its counts are written beside the pairs (`teacher-train+teacher-train-2-doubt0.5.stats.json`): the totals, then each run's sessions, `kept`, `doubted` and `frozen_content`. Each pairs file is built once and kept on Drive; the transcripts it is built from are the teacher runs', so no teacher is needed.

In [ ]:
!pip install -q -U transformers peft flash-linear-attention
!pip uninstall -q -y torchao
from huggingface_hub import snapshot_download

BASE = "/content/models/qwen35-08b"
snapshot_download(STUDENT, local_dir=BASE)
%cd {EVAL}
if not os.path.exists(PAIRS):
    sh(["python3", "sft_data.py", *TEACHER_RUNS, "--template", BASE, "--tokenizer", BASE,
                    "--verify-template", "--max-len", str(MAX_LEN), "--out", PAIRS,
                    *(["--doubt-confidence", str(DOUBT_CONFIDENCE)] if DOUBT_CONFIDENCE is not None else [])])
print(open(PAIRS.replace(".jsonl", ".stats.json")).read())

## SFT

One sequence per pair, eight to an optimizer step, the loss on the completion only, averaged as `LOSS` says (`train_args.json` beside the adapter records it). It saves a checkpoint every 50 steps on Drive and resumes from it when rerun. Each step prints its loss by kind and the tokens a second.

In [ ]:
%cd {EVAL}
if not os.path.exists(f"{ADAPTER}/adapter_model.safetensors"):
    sh(["python3", "train_sft.py", "--model", BASE, "--data", PAIRS,
                    "--out", ADAPTER, "--epochs", str(EPOCHS), "--rank", str(RANK), "--lr", str(LR),
                    "--accum", str(ACCUM), "--max-len", str(MAX_LEN), "--loss", LOSS, "--save-every", "50"])
!tail -n 3 {ADAPTER}/train_log.jsonl

## Merge

The adapter is added into a copy of the downloaded checkpoint, which keeps its layout, its tokenizer and its chat template. The merged folder on Drive is what the T4 notebook converts to a GGUF.

In [ ]:
%cd {EVAL}
sh(["python3", "merge_lora.py", "--base", BASE, "--adapter", ADAPTER, "--out", MERGED])
!ls -la {MERGED}

## Optional: the merged model unquantized on `dev` and `dev-novel`

About 4 minutes once the server is up. The same harness as the 0.8B baselines, on vLLM at bf16, so this row says what SFT did before any quantization; the T4 GGUF runs then say what 8 and 4 bits cost on top. Neither suite is trained on. Each suite is compared with the previous round, GRPO round 2 and the teacher (first named = baseline), and the tool rounds a case are listed. Each run also prints its mean confidence on right and on wrong drafts and the share of the drafts it would auto-approve that are right (the approval precision; the reward is built around 0.9).

In [ ]:
serve(MERGED, "--dtype bfloat16")
os.chdir(EVAL)
RUN_ARGS = ["--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen", "--reasoning-effort", "none", "--workers", "16"]
sh(["python3", "run.py", "--suite", "dev", "--name", f"{NAME}-bf16", *RUN_ARGS])
sh(["python3", "run.py", "--suite", "dev-novel", "--name", f"{NAME}-bf16-dev-novel", *RUN_ARGS])
stop_servers()
for lineup in ([f"{PREVIOUS}-bf16", f"{GRPO_REFERENCE}-bf16", f"{NAME}-bf16", TEACHER_NAME],
               [f"{PREVIOUS}-bf16-dev-novel", f"{GRPO_REFERENCE}-bf16-dev-novel", f"{NAME}-bf16-dev-novel",
                TEACHER_NOVEL_RUN]):
    present = [run for run in lineup if finished(f"{RECORDS_DIR}/runs/{run}", 280)]
    sh(["python3", "compare.py", *present])
    for run in present:
        with open(f"{RECORDS_DIR}/runs/{run}/grade.json") as f:
            process = json.load(f)["summary"]["process"]
        print(f"{run}: {process['rounds']:.2f} model rounds a case, tool calls a case by name",
              {name: round(n, 2) for name, n in process["tool_calls_by_name"].items()})
        with open(f"{RECORDS_DIR}/runs/{run}/grade.json") as f:
            cases = json.load(f)["per_case"]
        cases = list(cases.values()) if isinstance(cases, dict) else cases
        drafts = [c for c in cases if c["answered"] and isinstance(c.get("confidence"), (int, float))]
        right = [c["confidence"] for c in drafts if c["record_all"]]
        wrong = [c["confidence"] for c in drafts if not c["record_all"]]
        approved = [c for c in cases if c["would_approve"]]
        print(f"   confidence {sum(right) / max(1, len(right)):.3f} on right drafts, {sum(wrong) / max(1, len(wrong)):.3f} "
              f"on wrong ones; approval precision {sum(c['record_all'] for c in approved) / max(1, len(approved)):.2f} "
              f"over {len(approved)} approved")
sh(["python3", "board.py", "--write"])

## Take the results home

The zip holds what is worth committing: each teacher run's `meta.json`, `results.jsonl` and `grade.json` (its transcripts stay on Drive: they are the training material, tens of megabytes), the pair counts, the adapter's config and train log, the merge report, the teacher's `dev-novel` run, the bf16 runs if there are any, and `RESULTS.md`. Unpack it into your records folder.

In [ ]:
import shutil

OUT = "/content/out"
shutil.rmtree(OUT, ignore_errors=True)
for run, folder in TEACHER_DIRS.items():
    os.makedirs(f"{OUT}/runs/{run}")
    for name in ("meta.json", "results.jsonl", "grade.json"):
        shutil.copy(f"{folder}/{name}", f"{OUT}/runs/{run}/")
os.makedirs(f"{OUT}/sft/{NAME}")
for source in (PAIRS.replace(".jsonl", ".stats.json"), f"{ADAPTER}/adapter_config.json", f"{ADAPTER}/train_args.json",
               f"{ADAPTER}/train_log.jsonl", f"{MERGED}/merge.json"):
    shutil.copy(source, f"{OUT}/sft/{NAME}/")
for run in (TEACHER_NOVEL_RUN, f"{NAME}-bf16", f"{NAME}-bf16-dev-novel"):
    if finished(f"{RECORDS_DIR}/runs/{run}", 1):
        shutil.copytree(f"{RECORDS_DIR}/runs/{run}", f"{OUT}/runs/{run}")
shutil.copy(f"{RECORDS_DIR}/RESULTS.md", OUT)
subprocess.run(f"cd {OUT} && zip -qr /content/{NAME}.zip .", shell=True, check=True)
from google.colab import files
files.download(f"/content/{NAME}.zip")